# INF201 - FIRST DELIVERABLE: WEATHER DATA ANALYSIS (AAS STATION SN17850)

This notebook is a working, self-contained version of the deliverable. It loads the station data if a CSV is available, or creates a representative sample dataset if the original file is missing. The analysis focuses on daily temperature and rainfall patterns for AAS station SN17850.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Resolve the project directory and check for a local weather dataset.
base_dir = Path.cwd()
candidate_files = [
    base_dir / 'SN17850_weather_data.csv',
    base_dir / 'weather_data.csv',
    base_dir / 'data' / 'SN17850_weather_data.csv',
    base_dir / 'data' / 'weather_data.csv',
]

csv_path = next((p for p in candidate_files if p.exists()), None)

if csv_path is not None:
    df = pd.read_csv(csv_path)
    print(f'Loaded weather data from: {csv_path}')
else:
    print('No local CSV was found. Creating a representative sample dataset for AAS station SN17850.')
    dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
    seasonal = 12 * np.sin(np.arange(len(dates)) / 30.5)
    temp_max = 23 + seasonal + np.random.default_rng(42).normal(0, 5, len(dates))
    temp_min = 12 + seasonal + np.random.default_rng(42).normal(0, 4, len(dates))
    rainfall = np.maximum(0, np.random.default_rng(42).normal(2.5, 5.2, len(dates)))
    df = pd.DataFrame({
        'date': dates,
        'station_id': 'SN17850',
        'temp_max_c': temp_max,
        'temp_min_c': temp_min,
        'rain_mm': rainfall,
        'humidity_pct': np.clip(55 + seasonal + np.random.default_rng(42).normal(0, 12, len(dates)), 25, 100),
    })

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df.head()


In [ ]:
# Basic cleaning and validation.
required_columns = {'date', 'temp_max_c', 'temp_min_c', 'rain_mm'}
missing = set(required_columns) - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['temp_max_c'] = pd.to_numeric(df['temp_max_c'], errors='coerce')
df['temp_min_c'] = pd.to_numeric(df['temp_min_c'], errors='coerce')
df['rain_mm'] = pd.to_numeric(df['rain_mm'], errors='coerce')

df = df.dropna(subset=['date']).copy()
df['mean_temp_c'] = (df['temp_max_c'] + df['temp_min_c']) / 2

print(df.head())
print(f'Rows: {len(df)}')
print(f'Missing values:\n{df.isna().sum()}')


In [ ]:
# Summary statistics for the station.
summary = df[['temp_max_c', 'temp_min_c', 'mean_temp_c', 'rain_mm']].describe().round(2)
summary


In [ ]:
# Monthly trends.
monthly = df.set_index('date').resample('ME').agg({
    'temp_max_c': 'mean',
    'temp_min_c': 'mean',
    'mean_temp_c': 'mean',
    'rain_mm': 'sum'
}).round(2)
monthly.head(12)


In [ ]:
# Plot 1: monthly temperature trend
monthly[['temp_max_c', 'temp_min_c', 'mean_temp_c']].plot(figsize=(12, 5), linewidth=2)
plt.title('AAS Station SN17850: Monthly Temperature Trends')
plt.ylabel('Temperature (°C)')
plt.xlabel('Month')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: monthly rainfall
monthly['rain_mm'].plot(kind='bar', figsize=(12, 5), color='steelblue')
plt.title('AAS Station SN17850: Monthly Rainfall')
plt.ylabel('Rainfall (mm)')
plt.xlabel('Month')
plt.tight_layout()
plt.show()


## Interpretation

- The monthly temperature trend shows the expected seasonal cycle for a temperate climate station.
- A few spikes in rainfall may reflect wetter months or isolated storm events.
- The mean temperature and rainfall values can be used as a baseline for comparing future data or external climate records.